# 04 · Tier 1: hand-built graph features into XGBoost

One unit per dataset × graph builder × seed. This is the version of "use the graph" a bank could implement with
SQL: degree, two-hop reach, edge-type counts, neighbour feature means, and the neighbour default rate computed from
**training labels only**. The features are appended to the tabular matrix and the same XGBoost recipe as Tier 0 is fit.

The comparison that matters commercially is Tier 1 versus Tier 3 (notebook 06): how much of the graph lift needs a
neural network at all?

Test rows are attached through query edges (test → train/val) so their neighbourhood features exist; test labels are
never read. On DGraph-Fin the real graph is used as-is.

**How this notebook runs.** Never by hand in an editor. `scripts/run_nb.sh 04_graph_features_xgb dataset=... seed=...` executes it
with papermill on the GPU node, writes the executed copy to `runs/04_graph_features_xgb/<dataset>/<graph_builder>/<seed>/executed.ipynb`,
and the last cell writes a `DONE` marker. If the process dies, the wrapper writes `FAILED` and the next `run_stage.sh`
picks the unit up again; checkpoints inside the run directory mean little work is repeated.

In [ ]:
# Parameters (papermill overrides these; keep names and types)
dataset = "taiwan"
seed = 0
graph_builder = "knn"
run_dir = ""       # empty -> runs/<stage>/<dataset>/<graph_builder>/<seed>
force = 0          # 1 -> redo work even if checkpoints exist
quick = 0          # 1 -> subsampled data, few epochs; smoke test only

In [ ]:
import json
import shutil
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import ckpt, config, paths, viz
from src.eval import results

STAGE = "04_graph_features_xgb"
warnings.filterwarnings("ignore")
seed = int(seed)
force, quick = bool(int(force)), bool(int(quick))
run_dir = Path(run_dir) if run_dir else paths.run_dir(STAGE, dataset, graph_builder, seed)
run_dir.mkdir(parents=True, exist_ok=True)
t0 = time.time()
print(f"stage={STAGE} dataset={dataset} graph_builder={graph_builder} seed={seed} quick={quick} force={force}")
print("run_dir:", run_dir)
pd.Series(paths.describe())

In [ ]:
from src.data import splits
from src.data.base import Bundle

cfg = config.load_config(dataset)
models_cfg = config.quick_override(cfg, quick)
bundle = Bundle.load(paths.processed_dir(dataset, quick))
split = splits.load_split(dataset, quick)
X, y = bundle.X, bundle.y.to_numpy()
tr, va, te = split["train"], split["val"], split["test"]
allowed = np.sort(np.concatenate([tr, va]))          # rows that may take part in graph construction
X_tr, X_va, X_te = X.iloc[tr], X.iloc[va], X.iloc[te]
y_tr, y_va, y_te = y[tr], y[va], y[te]
gdir = paths.graphs_dir(dataset, quick)
n_boot = 200 if quick else cfg["bootstrap"]["n_boot"]
print(bundle.summary())
print({p: len(split[p]) for p in splits.PARTS}, "| test positive rate:", round(float(np.nanmean(y_te)), 4))

## Load the graph and compute features

In [ ]:
from src.data.splits import masks
from src.graphs import load_edges
from src.graphs.features import graph_features

edges = load_edges(dataset, graph_builder, cfg, quick, variant="full")
print("edges:", edges.n_edges, "| meta:", {k: v for k, v in edges.meta.items() if k != "stats"})
train_mask = masks(split, bundle.n)["train"]
t = time.time()
G = graph_features(edges, X.to_numpy(np.float32), y, train_mask, feature_cols=list(X.columns))
print(f"{G.shape[1]} graph features in {time.time() - t:.0f}s")
deg_te = G["g_degree"].to_numpy()[te]
print(f"test rows with at least one edge: {np.mean(deg_te > 0):.1%}")
G.describe().T.head(12)

## Fit XGBoost on tabular + graph features

In [ ]:
from src.eval import metrics
from src.models import xgb as XGB

XG = pd.concat([X.reset_index(drop=True), G.reset_index(drop=True)], axis=1)
xgb_params = dict(models_cfg["xgb"])
t = time.time()
booster = XGB.fit_xgb(XG.iloc[tr], y_tr, XG.iloc[va], y_va, xgb_params, seed=seed)
p_va, p_te = XGB.predict_xgb(booster, XG.iloc[va]), XGB.predict_xgb(booster, XG.iloc[te])
timing = round(time.time() - t, 1)
XGB.save_xgb(booster, run_dir / "models" / "xgb_graphfeat.json")
print(f"val AUC {metrics.auc(y_va, p_va):.4f}   test AUC {metrics.auc(y_te, p_te):.4f}   {timing}s")

## Against the Tier 0 XGBoost

Same dataset, same seed, same test rows, paired bootstrap.

In [ ]:
base = results.baseline_predictions(dataset, seed, "xgb", "test")
compare = results.compare_against(base, te, y_te, {"xgb_graphfeat": p_te}, n_boot=n_boot, seed=seed)
cmp_df = pd.DataFrame(compare)
fig = viz.plot_intervals(cmp_df, label="metric", title=f"Tier 1 gain over XGBoost ({dataset}/{graph_builder}, seed {seed})")
viz.savefig(fig, run_dir, "gain_vs_xgb")
plt.show()
cmp_df[["metric", "base", "new", "diff", "ci_low", "ci_high", "significant"]].round(4)

## Which graph features matter

In [ ]:
imp = XGB.importance(booster, "gain")
g_imp = imp[imp.index.str.startswith("g_")]
print(f"graph features share of total gain: {g_imp.sum() / imp.sum():.1%}")
fig = viz.plot_bar(g_imp.head(20), title="Graph feature gain (top 20)", ylabel="gain")
viz.savefig(fig, run_dir, "graph_feature_importance")
plt.show()
_, p_base, _ = results.align(base, pd.DataFrame({"position": te, "y": y_te, "p": p_te}))
fig = viz.plot_roc_pr(y_te, {"xgb (tier 0)": p_base, "xgb + graph features": p_te}, title=f"{dataset} test")
viz.savefig(fig, run_dir, "roc_pr")
plt.show()

In [ ]:
results.save_predictions(run_dir, "xgb_graphfeat", {"val": (va, y_va, p_va), "test": (te, y_te, p_te)})
rows = {"xgb_graphfeat": results.metrics_block(y_va, p_va, y_te, p_te)}
results.save_metrics(run_dir, {"models": rows, "compare": compare,
                               "params": {"graph_builder": graph_builder, "seed": seed, "quick": quick, "n_graph_features": int(G.shape[1]),
                                          "graph_gain_share": float(g_imp.sum() / imp.sum()), "timing_s": timing, "xgb": XGB.summary(booster)}})
summary = {"test_auc": round(rows["xgb_graphfeat"]["test"]["auc"], 4), "auc_diff_vs_xgb": round(cmp_df.loc[cmp_df.metric == "auc", "diff"].iloc[0], 4),
           "elapsed_s": round(time.time() - t0, 1)}
ckpt.mark_done(run_dir, summary)
summary